## Begin ##

First load the dataset - fra_cleaned.csv.  df.shape prints the rows and columns of the dataset, and df.head displays the first five entries.

In [ ]:
df = pd.read_csv('fra_cleaned.csv', encoding='latin-1', sep=';')
print(df.shape)
print(df.head())

(24063, 18)
                                                 url  \
0  https://www.fragrantica.com/perfume/xerjoff/ac...   
1  https://www.fragrantica.com/perfume/jean-paul-...   
2  https://www.fragrantica.com/perfume/jean-paul-...   
3  https://www.fragrantica.com/perfume/bruno-bana...   
4  https://www.fragrantica.com/perfume/jean-paul-...   

                          Perfume               Brand  Country  Gender  \
0  accento-overdose-pride-edition             xerjoff    Italy  unisex   
1            classique-pride-2024  jean-paul-gaultier   France   women   
2            classique-pride-2023  jean-paul-gaultier   France  unisex   
3               pride-edition-man        bruno-banani  Germany     men   
4         le-male-pride-collector  jean-paul-gaultier   France     men   

  Rating Value  Rating Count    Year  \
0         1,42           201  2022.0   
1         1,86            70  2024.0   
2         1,91           285  2023.0   
3         1,92            59  2019.0   
4     

## Get all notes ##

The next step is to build the fragrance matrix - First, we will combine all the notes from the top, middle, and base notes columns into one 'set' per fragrance.  This gives each fragrance a list like ['vanilla', 'amber', 'vetiver']

In [ ]:
def get_all_notes(row):
    notes = []
    for col in ['Top', 'Middle', 'Base']:
        if pd.notna(row[col]):
            notes.extend([n.strip().lower() for n in row[col].split(',')])
    return notes

df['all_notes'] = df.apply(get_all_notes, axis=1)

print(df[['Perfume', 'all_notes']].head()) # Prints first 5 fragrances with note lists

                          Perfume  \
0  accento-overdose-pride-edition   
1            classique-pride-2024   
2            classique-pride-2023   
3               pride-edition-man   
4         le-male-pride-collector   

                                           all_notes  
0  [fruity notes, aldehydes, green notes, bulgari...  
1  [yuzu, citruses, orange blossom, neroli, musk,...  
2  [blood orange, yuzu, neroli, orange blossom, m...  
3  [guarana, grapefruit, red apple, walnut, laven...  
4  [mint, lavender, cardamom, artemisia, bergamot...  


## How many total notes? ##

This counts the unique notes in the dataset.

In [ ]:
unique_notes = set()
for notes_list in df['all_notes']:
    unique_notes.update(notes_list)

print(f"Unique notes: {len(unique_notes)}")
print(f"\nFirst 10 (alphabetically): {sorted(unique_notes)[:10]}")

Unique notes: 1671

First 10 (alphabetically): ['absinthe', 'acai berry', 'accord eudora®', 'acerola', 'acerola blossom', 'acetylfuran', 'acácia', 'african freesia petals', 'african geranium', 'african ginger']


## Building the matrix ##

Messing around with populating the matrix using a dictionary data structure and numbers for column names vs just having the column name as the note.

TODO: bring up how creating a matrix at this step is a lot simpler than comparing string literals to each other

In [ ]:
import numpy as np
import time

sorted_notes = sorted(unique_notes)

# --- O(n) approach: list scanning ---
start = time.time()

matrix_slow = np.zeros((len(df), len(sorted_notes)), dtype=int)

for row_idx, notes_list in enumerate(df['all_notes']):
    for note in notes_list:
        col_idx = sorted_notes.index(note)  # scans the list every time
        matrix_slow[row_idx, col_idx] = 1

slow_time = time.time() - start
print(f"List approach (O(n) lookup): {slow_time:.2f} seconds")

# --- O(1) approach: dictionary ---
start = time.time()

note_to_index = {note: i for i, note in enumerate(sorted_notes)}
matrix_fast = np.zeros((len(df), len(note_to_index)), dtype=int)

for row_idx, notes_list in enumerate(df['all_notes']):
    for note in notes_list:
        col_idx = note_to_index[note]  # direct hash lookup
        matrix_fast[row_idx, col_idx] = 1

fast_time = time.time() - start
print(f"Dict approach (O(1) lookup): {fast_time:.2f} seconds")

print(f"\nSpeedup: {slow_time / fast_time:.1f}x faster")
print(f"Matrices match: {np.array_equal(matrix_slow, matrix_fast)}")

List approach (O(n) lookup): 1.25 seconds
Dict approach (O(1) lookup): 0.05 seconds

Speedup: 25.0x faster
Matrices match: True


## O(n) vs. O(1) approaches ##

As we can see, the list approach is 25x faster - for a little over 24,000 fragrances, that gives us 1.25 seconds for the O(n) approach and 0.05 seconds for the O(1) approach!  The matrices are the exact same, but one was created 25x faster!

Though I only saved a little over a second with a dataset of this size, O(n) scales linearly with the number of features so it would only keep getting worse.

Let's make matrix_fast our note_matrix for the rest of the analysis and continue.

In [ ]:
note_matrix = matrix_fast

print(f"Matrix shape: {note_matrix.shape}")
print(f"  {note_matrix.shape[0]} fragrances x {note_matrix.shape[1]} notes\n")

# How many notes does each fragrance have on average?
notes_per_frag = note_matrix.sum(axis=1)
print(f"Notes per fragrance - min: {notes_per_frag.min()}, max: {notes_per_frag.max()}, avg: {notes_per_frag.mean():.1f}\n")

# How sparse is the matrix? (mostly zeros)
total_cells = note_matrix.shape[0] * note_matrix.shape[1]
filled_cells = note_matrix.sum()
print(f"Sparsity: {filled_cells}/{total_cells} cells filled ({filled_cells/total_cells*100:.2f}%)\n")

# Spot check a known fragrance
idx = df[df['Perfume'].str.contains('shalimar', case=False)].index[0]
frag_notes = [note for note, col in note_to_index.items() if note_matrix[idx, col] == 1]
print(f"Spot check - {df.iloc[idx]['Perfume']}:")
print(f"  Notes: {frag_notes}")

Matrix shape: (24063, 1671)
  24063 fragrances x 1671 notes

Notes per fragrance - min: 1, max: 62, avg: 9.8

Sparsity: 236985/40209273 cells filled (0.59%)

Spot check - shalimar-souffle-d-oranger:
  Notes: ['bergamot', 'jasmine sambac', 'mandarin orange', 'neroli', 'orange blossom', 'petitgrain', 'sandalwood', 'vanilla']


We got the matrix shape and a bit more info about our matrix. 24063 fragrances x 1671 notes.
The average fragrance has 9.8 notes out of a potential 1671!
This is a very sparse matrix.

For the next part, I will act as the user selecting a fragrance - In this case, 'Dakar' by Thera Cosmeticos.

In [ ]:
search = df[df['Perfume'].str.contains('dakar', case=False)][['Perfume', 'Brand']]
print(search)

     Perfume             Brand
4119   dakar  thera-cosmeticos


We found the index of the fragrance, which we need for the next step.

In [ ]:
target_idx = 4119
target_row = matrix_fast[target_idx]

# Jaccard: intersection / union for each fragrance
scores = []
for i in range(len(matrix_fast)):
    if i == target_idx:
        continue
    intersection = np.sum((target_row == 1) & (matrix_fast[i] == 1))
    union = np.sum((target_row == 1) | (matrix_fast[i] == 1))
    score = intersection / union if union > 0 else 0
    scores.append((i, score))

# Sort by score descending, grab top 10
scores.sort(key=lambda x: x[1], reverse=True)

print(f"Recommendations for: {df.iloc[target_idx]['Perfume']} by {df.iloc[target_idx]['Brand']}\n")
print(f"Its notes: {df.iloc[target_idx]['all_notes']}\n")
print("Top 10 Jaccard matches:")
for rank, (idx, score) in enumerate(scores[:10], 1):
    print(f"  {rank}. {df.iloc[idx]['Perfume']} by {df.iloc[idx]['Brand']} — {score:.3f}")
    print(f"     Notes: {df.iloc[idx]['all_notes']}")

Recommendations for: dakar by thera-cosmeticos

Its notes: ['spicy notes', 'tobacco leaf', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao', 'woody notes', 'dried fruits']

Top 10 Jaccard matches:
  1. tobacco-vanille by tom-ford — 1.000
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'cacao', 'tonka bean', 'tobacco blossom', 'dried fruits', 'woody notes']
  2. sweetobacco by in-the-box — 0.778
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tobacco blossom', 'cacao', 'tonka bean', 'woodsy notes', 'dried fruits']
  3. efrate by nuancielo — 0.778
     Notes: ['spicy notes', 'tobacco leaf', 'vanilla', 'cacao', 'tobacco blossom', 'tonka bean', 'dried fruits', 'woodsy notes']
  4. vanille-persuasive by lpdo — 0.778
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao pod', 'dried fruits', 'woody notes']
  5. tabac-gourmand by patrice-martin — 0.778
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tonka bean', 'tobacco bloss

## END PART 1 OF ANALYSIS ##

Jaccard performed well as a simple test, matching Fragrantica's similar fragrances for Dakar.  But it has some limitations: literal notes overlap, and fragrances with different numbers of notes may be down weighted in this kind of analysis.